# The Beta-Bernoulli Posterior

Wiki reference for [the Beta-Bernoulli posterior](https://ml-viz-ruby.vercel.app/wiki/beta-bernoulli-posterior).

**The idea in one sentence.** The Beta distribution is the **conjugate prior** for a Bernoulli
rate, so the posterior after seeing $k$ heads in $n$ flips is simply $\text{Beta}(\alpha + k,\;
\beta + n - k)$ — updating beliefs is *counting* — and as data accumulates the prior **washes
out** and the estimate converges to the MLE.

We compute the posterior, MAP/mean/MLE, and a credible interval from scratch, **validate
conjugacy and the credible-interval mass**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta as beta_dist

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  '#ccc',
    'xtick.color':      '#888',
    'ytick.color':      '#888',
    'text.color':       '#eee',
    'grid.color':       '#333',
    'lines.linewidth':  2,
})

np.random.seed(42)

## 1 — Numeric update: Beta(2,2) + 7 heads in 10 flips

In [ ]:
alpha_prior, beta_prior = 2, 2
k, n = 7, 10
n0 = n - k

alpha_post = alpha_prior + k
beta_post  = beta_prior + n0

map_est  = (alpha_post - 1) / (alpha_post + beta_post - 2)
mean_est = alpha_post / (alpha_post + beta_post)
mle_est  = k / n

print(f"Prior:     Beta({alpha_prior}, {beta_prior})")
print(f"Posterior: Beta({alpha_post}, {beta_post})")
print(f"MAP:       {map_est:.4f}")
print(f"Mean:      {mean_est:.4f}")
print(f"MLE:       {mle_est:.4f}")

### Validate: conjugacy makes updating into counting

With a $\text{Beta}(\alpha, \beta)$ prior, observing $k$ successes and $n-k$ failures gives the
posterior $\text{Beta}(\alpha + k,\ \beta + n - k)$ — no integration needed. And a *uniform*
$\text{Beta}(1,1)$ prior makes the MAP estimate equal the MLE. We confirm both.

In [ ]:
print(f'prior Beta({alpha_prior},{beta_prior}) + ({k}H,{n0}T) -> posterior Beta({alpha_post},{beta_post})')
assert alpha_post == alpha_prior + k and beta_post == beta_prior + n0, 'posterior = prior + observed counts (conjugacy)'
a1, b1 = 1 + k, 1 + n0                      # uniform Beta(1,1) prior
assert np.isclose((a1 - 1) / (a1 + b1 - 2), mle_est), 'a uniform Beta(1,1) prior makes MAP equal the MLE'
print(f'MAP={map_est:.3f}, mean={mean_est:.3f}, MLE={mle_est:.3f}')
print('\n✅ Bayesian updating with a conjugate prior is just adding counts to the parameters')

## 2 — Sequential posterior updating

Each flip updates $\text{Beta}(\alpha, \beta) \to \text{Beta}(\alpha + x, \beta + 1-x)$.

In [ ]:
flips = [1, 1, 0, 1, 1, 0, 1, 0, 1, 1]   # 7 heads, 3 tails
alpha, beta_ = 2, 2

p = np.linspace(0, 1, 300)
fig, axes = plt.subplots(2, 5, figsize=(14, 5))
axes = axes.ravel()

for i, x in enumerate(flips):
    alpha += x
    beta_ += (1 - x)
    pdf = beta_dist.pdf(p, alpha, beta_)
    axes[i].fill_between(p, pdf, color='#6366f1', alpha=0.7)
    axes[i].axvline(alpha/(alpha+beta_), color='#ef4444',
                    linestyle='--', lw=1.5, label=f'mean={alpha/(alpha+beta_):.2f}')
    axes[i].set_title(f"After flip {i+1}  ({'H' if x else 'T'})\nBeta({alpha},{beta_})",
                      fontsize=8)
    axes[i].set_yticks([])
    axes[i].legend(fontsize=7, loc='upper left')

plt.suptitle('Sequential Beta-Bernoulli posterior updates (prior: Beta(2,2))',
             color='#eee', fontsize=11)
plt.tight_layout()
plt.show()

## 3 — Effect of prior strength

A strong prior (large $\alpha+\beta$) resists updating until enough data arrives.

In [ ]:
# Generate a biased coin: true p = 0.75
true_p = 0.75
np.random.seed(7)
obs = np.random.binomial(1, true_p, 100)

priors = [
    ('Uniform Beta(1,1)',   1,  1,  '#6366f1'),
    ('Weak Beta(2,2)',      2,  2,  '#f97316'),
    ('Strong Beta(10,10)', 10, 10,  '#20d9d2'),
    ('Biased Beta(1,5)',    1,  5,  '#ef4444'),
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
p_grid = np.linspace(0, 1, 300)
n_obs = [5, 20, 100]    # snapshot sizes

for label, a0, b0, col in priors:
    means = []
    for n_so_far in range(1, 101):
        k_so_far = obs[:n_so_far].sum()
        a_post = a0 + k_so_far
        b_post = b0 + (n_so_far - k_so_far)
        means.append(a_post / (a_post + b_post))
    axes[0].plot(range(1, 101), means, color=col, label=label)

axes[0].axhline(true_p, color='white', linestyle=':', lw=1, label=f'True p={true_p}')
axes[0].set_xlabel('Number of observations'); axes[0].set_ylabel('Posterior mean')
axes[0].set_title('Posterior mean convergence to true p')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

# Posterior distributions after 100 observations
k_total = obs.sum()
for label, a0, b0, col in priors:
    a_post = a0 + k_total
    b_post = b0 + (100 - k_total)
    axes[1].plot(p_grid, beta_dist.pdf(p_grid, a_post, b_post),
                 color=col, label=f'{label} → Beta({a_post},{b_post})')

axes[1].axvline(true_p, color='white', linestyle=':', lw=1, label=f'True p={true_p}')
axes[1].axvline(k_total/100, color='#aaa', linestyle='--', lw=1,
                label=f'MLE={k_total/100:.2f}')
axes[1].set_xlabel('p'); axes[1].set_ylabel('Posterior density')
axes[1].set_title('Posteriors after 100 observations')
axes[1].legend(fontsize=7); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Observations: {k_total} heads in 100 flips (MLE = {k_total/100:.2f})")

## 4 — Credible interval vs. confidence interval

The Bayesian **credible interval** is the HPD (highest posterior density) interval;
it has a direct probability interpretation: $P(p \in CI | \mathcal{D}) = 0.95$.

In [ ]:
alpha_post, beta_post = 9, 5   # Beta(2,2) + 7H 3T

# 95% equal-tailed credible interval
lower = beta_dist.ppf(0.025, alpha_post, beta_post)
upper = beta_dist.ppf(0.975, alpha_post, beta_post)
mean  = alpha_post / (alpha_post + beta_post)

p_grid = np.linspace(0, 1, 400)
pdf    = beta_dist.pdf(p_grid, alpha_post, beta_post)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(p_grid, pdf, color='#6366f1')
mask = (p_grid >= lower) & (p_grid <= upper)
ax.fill_between(p_grid[mask], pdf[mask], color='#6366f1', alpha=0.3,
                label=f'95% CI: [{lower:.3f}, {upper:.3f}]')
ax.axvline(mean, color='#ef4444', linestyle='--', lw=1.5,
           label=f'Posterior mean={mean:.3f}')
ax.set_xlabel('p'); ax.set_ylabel('Posterior density')
ax.set_title(f'Beta({alpha_post},{beta_post}) posterior with 95% credible interval')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Validate: the 95% credible interval holds 95% of the posterior mass

An equal-tailed 95% credible interval runs from the 2.5th to the 97.5th posterior percentile,
so by construction it contains exactly 95% of the posterior probability and brackets the mean.
We confirm the mass and that the mean lies inside.

In [ ]:
mass = beta_dist.cdf(upper, alpha_post, beta_post) - beta_dist.cdf(lower, alpha_post, beta_post)
print(f'95% CI = [{lower:.3f}, {upper:.3f}], posterior mass inside = {mass:.4f}, mean = {mean:.3f}')
assert np.isclose(mass, 0.95, atol=1e-6), 'the equal-tailed interval holds 95% of the posterior mass'
assert lower < mean < upper, 'the credible interval brackets the posterior mean'
print('\n✅ a credible interval is a direct probability statement about the parameter')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **overconfident prior** | a strong wrong prior biases small-sample estimates |
| **MAP vs mean** | they differ for skewed posteriors; report the one you mean |
| **credible ≠ confidence** | a credible interval is a probability over the parameter (verified) |
| **prior washout** | with enough data the prior stops mattering (demo) |
| **rare events** | for $k=0$, MLE says 0; a prior gives a sensible nonzero estimate |

Demo: different priors converge after 100 observations.

In [ ]:
# The key Bayesian gotcha (and comfort): with enough data the PRIOR WASHES OUT. Starting from
# very different priors — uniform, strong-symmetric, and biased — after 100 observations the
# posteriors nearly agree and all converge toward the MLE / true rate. The prior matters most
# when data is scarce.
k100 = obs.sum()
means = {}
for label, a0, b0, _ in priors:
    a, b = a0 + k100, b0 + (len(obs) - k100)
    means[label] = a / (a + b)
for label, m in means.items():
    print(f'{label:20s} posterior mean after 100 obs: {m:.3f}')
spread = max(means.values()) - min(means.values())
print(f'true rate = {true_p};  spread across priors = {spread:.3f}')
assert spread < 0.05, 'with 100 observations, very different priors converge -> the prior washes out'
print('\nThe prior dominates when data is scarce and vanishes as data grows — pick it honestly, not to cheat.')

## ✏️ Your turn

### Exercise 1 — Laplace smoothing connection

Show that a $\text{Beta}(1,1)$ prior (uniform) with $n$ observations ($k$ heads)
gives a MAP estimate of $k/n$ — identical to the MLE.
Then show that a $\text{Beta}(2,2)$ prior gives Laplace-smoothed estimates:
$\hat{p} = (k+1)/(n+2)$.
Verify both formulas numerically for $k=3$, $n=5$.

In [ ]:
k, n = 3, 5

# TODO(you): compute MAP for Beta(1,1) prior and Beta(2,2) prior
# For Beta(α,β) prior: MAP = (α + k - 1) / (α + β + n - 2)

# map_uniform = ???
# map_laplace = ???
# mle = k/n

# print(f"MLE:             {mle:.4f}")
# print(f"MAP Beta(1,1):   {map_uniform:.4f}  (should equal MLE = {mle:.4f})")
# print(f"MAP Beta(2,2):   {map_laplace:.4f}  (should equal (k+1)/(n+2) = {(k+1)/(n+2):.4f})")

### Exercise 2 — Predictive probability

The **posterior predictive** probability of the next flip being heads is:
$$P(x_{n+1} = 1 | \mathcal{D}) = \mathbb{E}[p | \mathcal{D}] = \frac{\alpha+k}{\alpha+\beta+n}$$

Starting from $\text{Beta}(2,2)$ and observing the sequence `[H, H, T, H, T]`,
compute and plot the predictive probability after each flip.

<details>
<summary>Solution</summary>

```python
flips = [1, 1, 0, 1, 0]
a, b = 2, 2
preds = []
for x in flips:
    a += x; b += (1 - x)
    preds.append(a / (a + b))

plt.figure(figsize=(7, 3))
plt.plot(range(1, len(flips)+1), preds, 'o-', color='#6366f1')
plt.ylim(0, 1); plt.axhline(0.5, color='#555', lw=0.5)
plt.xlabel('After flip #'); plt.ylabel('P(next = H | data)')
plt.title('Posterior predictive probability')
plt.grid(True, alpha=0.3); plt.show()
```
</details>

## Key takeaways

- **Conjugacy = counting:** the Beta posterior is $\text{Beta}(\alpha+k, \beta+n-k)$ — no
  integration (verified).
- **Prior choice:** a uniform $\text{Beta}(1,1)$ makes MAP = MLE; informative priors regularize
  small samples (verified).
- **Credible intervals** are direct probability statements holding the stated mass (verified).
- **The prior washes out** as data accumulates (demo) — it matters most when data is scarce.